In [ ]:
import os
import logging
from typing import List, Optional
import requests
from pymilvus import MilvusClient
from transformers import AutoTokenizer, AutoModelForCausalLM
from PyPDF2 import PdfReader
import pytesseract
from PIL import Image
import time
from sentence_transformers import SentenceTransformer
from openai import OpenAI 


from nv_ingest.framework.orchestration.ray.util.pipeline.pipeline_runners import run_pipeline
from nv_ingest.framework.orchestration.ray.util.pipeline.pipeline_runners import PipelineCreationSchema
from nv_ingest_client.client import Ingestor

# Setup logging
logging.basicConfig(level=logging.DEBUG, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Environment variables
if 'NVIDIA_API_KEY' not in os.environ:
    raise ValueError("NVIDIA_API_KEY not set.")
if 'HUGGINGFACE_TOKEN' not in os.environ:
    logger.warning("HUGGINGFACE_TOKEN not set; fallbacks may fail.")

NVIDIA_API_KEY = os.environ['NVIDIA_API_KEY']
HF_TOKEN = os.environ.get('HUGGINGFACE_TOKEN')

# Milvus lite client setup
MILVUS_DB_PATH = '/home/sneha-ltim/abrav/Document_Digitizer_backend/RAG_/milvus_rag.db'
try:
    milvus_client = MilvusClient(uri=MILVUS_DB_PATH)
    logger.info("Connected to Milvus lite client.")
except Exception as e:
    logger.error(f"Milvus lite client connection failed: {e}")
    raise

# Collection name
COLLECTION_NAME = 'rag_documents'

# NIM LLM setup
NIM_ENDPOINT = 'https://api.nvidia.com/v1/chat/completions'
HF_MODEL = 'meta-llama/Meta-Llama-3-8B-Instruct'

def get_llm_response(prompt: str) -> str:
    """Generate response using NVIDIA NIM or HuggingFace fallback."""
    headers = {'Authorization': f'Bearer {NVIDIA_API_KEY}', 'Content-Type': 'application/json'}
    data = {
        'model': 'meta/llama-3.2-90b-vision-instruct',
        'messages': [{'role': 'user', 'content': prompt}],
        'max_tokens': 200,
        'temperature': 0.7
    }
    try:
        response = requests.post(NIM_ENDPOINT, json=data, headers=headers)
        response.raise_for_status()
        return response.json()['choices'][0]['message']['content']
    except Exception as e:
        logger.error(f"NIM failed: {e}. Falling back to HuggingFace.")
        try:
            tokenizer = AutoTokenizer.from_pretrained(HF_MODEL, token=HF_TOKEN)
            model = AutoModelForCausalLM.from_pretrained(HF_MODEL, token=HF_TOKEN)
            inputs = tokenizer(prompt, return_tensors='pt')
            outputs = model.generate(**inputs, max_new_tokens=200)
            return tokenizer.decode(outputs[0], skip_special_tokens=True)
        except Exception as hf_e:
            logger.error(f"HF LLM failed: {hf_e}")
            return "Error: LLM generation failed."

def load_sentence_transformer_with_retry(model_name, retries: int = 3, delay: int = 5):
    """Load SentenceTransformer with retries."""
    for attempt in range(retries):
        try:
            model = SentenceTransformer(model_name)
            logger.info(f"Loaded SentenceTransformer model: {model_name}")
            return model
        except Exception as e:
            logger.error(f"Attempt {attempt + 1} failed to load model {model_name}: {e}. Retrying in {delay} seconds...")
            time.sleep(delay)
    raise Exception(f"Failed to load SentenceTransformer model {model_name} after {retries} attempts.")

def embed_text(texts: List[str], api_key: str, model_name: str = 'nvidia/nv-embedqa-e5-v5') -> List[List[float]]:
    """Embed texts using NVIDIA API or raise error for fallback."""
    headers = {'Authorization': f'Bearer {api_key}', 'Content-Type': 'application/json'}
    url = 'https://integrate.api.nvidia.com/v1/embeddings'
    data = {
        'model': model_name,
        'input': texts,
        'input_type': 'query'
    }
    try:
        response = requests.post(url, json=data, headers=headers)
        response.raise_for_status()
        embeddings = [emb['embedding'] for emb in response.json()['data']]
        return embeddings
    except Exception as e:
        logger.error(f"NVIDIA embedding failed: {e}")
        raise

def ingest_document(file_paths: List[str], output_dir: Optional[str] = None) -> List[dict]:
    """Ingest documents using NV-Ingest library mode with fallback."""
    logger.info(f"Ingesting documents: {file_paths}")
    results = []
    
    try:
        # Start pipeline in library mode (per ingest, non-persistent)
        config = PipelineCreationSchema()
        run_pipeline(config, block=False, disable_dynamic_scaling=True, run_in_subprocess=True)
        
        # Initialize Ingestor (no external client; local pipeline)
        ingestor = (
            Ingestor()
            .files(file_paths)
            .load()
            .extract(
                extract_text=True,
                extract_tables=True,
                extract_charts=True,
                extract_infographics=True,
                extract_images=True,
                table_output_format="markdown",
                text_depth="page"
            )
            .split(
                tokenizer="meta-llama/Llama-3.2-1B",
                chunk_size=512,
                chunk_overlap=50,
                params={"split_source_types": ["text", "table", "chart"], "hf_access_token": HF_TOKEN}
            )
            .caption(
                endpoint_url="https://integrate.api.nvidia.com/v1/chat/completions",
                model_name="nvidia/llama-3.1-nemotron-nano-vl-8b-v1",
                api_key=NVIDIA_API_KEY
            )
            .embed(
                endpoint_url="https://integrate.api.nvidia.com/v1",
                model_name="nvidia/nv-embedqa-e5-v5",
                api_key=NVIDIA_API_KEY
            )
        )
        
        if output_dir:
            ingestor = ingestor.save_to_disk(output_directory=output_dir, cleanup=True)
        
        # Upload to Milvus
        ingestor = ingestor.vdb_upload(
            collection_name=COLLECTION_NAME,
            milvus_uri=MILVUS_DB_PATH,
            dense_dim=1024 # Matches nv-embedqa-e5-v5
        )
        
        # Run ingestion
        logger.debug("Starting injestion process via NV-Ingest.")
        ingest_results, failures = ingestor.ingest(show_progress=True, return_failures=True)
        logger.debug("Ingestion process completed.")
        
        if failures:
            logger.warning(f"Ingestion failures: {len(failures)}. Details: {failures[:1]}")
        else:
            logger.info(f"Successfully ingested {len(ingest_results)} documents.")
        
        # Flatten results
        for doc_result in ingest_results:
            if hasattr(doc_result, 'chunks'):
                results.extend(doc_result.chunks)
            else:
                results.append({'text': doc_result.get('content', ''), 'embedding': doc_result.get('embedding', [])})
    
    except Exception as e:
        logger.warning(f"NV-Ingest failed: {e}. Falling back to custom ingestion.")
        for file_path in file_paths:
            try:
                text = ""
                if file_path.endswith('.pdf'):
                    reader = PdfReader(file_path)
                    text = ''.join(page.extract_text() or '' for page in reader.pages)
                elif file_path.endswith(('.jpg', '.png', '.jpeg')):
                    text = pytesseract.image_to_string(Image.open(file_path))
                elif file_path.endswith('.docx'):
                    from docx import Document
                    doc = Document(file_path)
                    text = '\n'.join(para.text for para in doc.paragraphs)
                elif file_path.endswith('.pptx'):
                    from pptx import Presentation
                    prs = Presentation(file_path)
                    text = '\n'.join(shape.text for slide in prs.slides for shape in slide.shapes if hasattr(shape, 'text'))
                elif file_path.endswith(('.html', '.md', '.txt', '.json', '.sh')):
                    with open(file_path, 'r') as f:
                        text = f.read()
                else:
                    logger.warning(f"Unsupported file type: {file_path}")
                    continue
                
                chunks = [text[i:i+512] for i in range(0, len(text), 462)]
                try:
                    embeddings = embed_text(chunks, NVIDIA_API_KEY)
                except:
                    logger.warning("NVIDIA embedding failed in fallback. Using SentenceTransformer.")
                    emb_model = load_sentence_transformer_with_retry('intfloat/e5-large')
                    embeddings = emb_model.encode(chunks).tolist()
                
                if not milvus_client.has_collection(COLLECTION_NAME):
                    milvus_client.create_collection(
                        collection_name=COLLECTION_NAME,
                        dimension=1024,
                        metric_type='L2'
                    )
                data = [{'text': chunk, 'embedding': emb} for chunk, emb in zip(chunks, embeddings)]
                milvus_client.insert(collection_name=COLLECTION_NAME, data=data)
                
                results.extend(data)
            except Exception as cust_e:
                logger.error(f"Custom ingestion failed for {file_path}: {cust_e}")
    
    stats = milvus_client.get_collection_stats(collection_name=COLLECTION_NAME)
    logger.info(f"Entities in Milvus after ingestion: {stats.get('row_count', 0)}")
    
    return results

def create_milvus_collection(dim: int = 1024):
    """Create Milvus collection if not exists."""
    try:
        if not milvus_client.has_collection(COLLECTION_NAME):
            milvus_client.create_collection(
                collection_name=COLLECTION_NAME,
                dimension=dim,
                metric_type='L2',
                auto_id=True,
            )
            logger.info("Milvus Lite collection created.")
        else:
            logger.info("Milvus collection already exists.")
    except Exception as e:
        logger.error(f"Failed to create Milvus collection: {e}")
        raise

def retrieve_context(query: str, top_k: int = 5) -> List[str]:
    """Retrieve relevant contexts from Milvus."""
    try:
        query_emb = embed_text([query], NVIDIA_API_KEY)[0]
        search_results = milvus_client.search(
            collection_name=COLLECTION_NAME,
            data=[query_emb],
            limit=top_k,
            output_fields=['text']
        )
        contexts = [hit['entity']['text'] for hit in search_results[0]]
        logger.debug(f"Retrieved {len(contexts)} contexts.")
        return contexts
    except Exception as e:
        logger.error(f"Retrieval failed: {e}. Falling back to HF embedder.")
        emb_model = load_sentence_transformer_with_retry('intfloat/e5-large')
        query_emb = emb_model.encode([query])[0].tolist()
        search_results = milvus_client.search(
            collection_name=COLLECTION_NAME,
            data=[query_emb],
            limit=top_k,
            output_fields=['text']
        )
        contexts = [hit['entity']['text'] for hit in search_results[0]]
        return contexts

def rag_chatbot(query: str) -> str:
    """RAG pipeline: Retrieve contexts and generate response."""
    contexts = retrieve_context(query)
    if not contexts:
        return "No relevant information found in the documents."
    
    context_str = '\n'.join(contexts)
    prompt = f"Context: {context_str}\n\nQuestion: {query}\nAnswer:"
    response = get_llm_response(prompt)
    logger.info(f"Generated response for query: {query}")
    return response

# Main execution for testing
if __name__ == "__main__":
    create_milvus_collection(dim=1024)
    
    # Test ingestion
    file_paths = ['/home/sneha-ltim/abrav/Document_Digitizer_backend/RAG_/Docs/BQ_GDD-000661395.pdf']
    # file_paths = ['./data/multimodal_test.pdf']
    output_dir = './temp_ingest_results'
    results = ingest_document(file_paths, output_dir=output_dir)
    
    print(f"Ingested {len(results)} chunks.")
    # Debug: Print first few chunks
    print("Sample chunks:", [r['text'][:100] + "..." for r in results[:2]])
    
    # Test RAG queries
    queries = [
        "What is the main topic of the document?"
        # Relevant for multimodal_test.pdf
    ]
    for query in queries:
        response = rag_chatbot(query)
        print(f"Query: {query}")
        print(f"Chatbot Response: {response}\n")


INFO:nv_ingest_api.util.system.hardware_info:Detected 8 logical cores via psutil.
INFO:nv_ingest_api.util.system.hardware_info:Detected 4 physical cores via psutil.
INFO:nv_ingest_api.util.system.hardware_info:Detected 8 cores via os.sched_getaffinity.
INFO:nv_ingest_api.util.system.hardware_info:Raw CPU limit determined: 8.00 (Method: sched_affinity)
INFO:nv_ingest_api.util.system.hardware_info:Applying hyperthread weight (0.75) to logical limit 8 (System: 4P/8L): Effective weighted cores = 7.00
INFO:nv_ingest_api.util.system.hardware_info:Effective CPU core limit determined: 7.00 (Method: sched_affinity_weighted)
INFO:nv_ingest_api.util.system.hardware_info:Detected 4 physical cores via psutil.
INFO:nv_ingest_api.util.system.hardware_info:Detected 8 cores via os.sched_getaffinity.
INFO:nv_ingest_api.util.system.hardware_info:Raw CPU limit determined: 8.00 (Method: sched_affinity)
INFO:nv_ingest_api.util.system.hardware_info:Applying hyperthread weight (0.75) to logical limit 8 (Syste

KeyboardInterrupt: 

INFO:nv_ingest.framework.orchestration.ray.primitives.ray_pipeline:Scaling/Maintenance stopped.
INFO:nv_ingest.framework.orchestration.ray.primitives.ray_stat_collector:Stopping stats collector thread...
INFO:nv_ingest.framework.orchestration.ray.primitives.ray_stat_collector:Stats collector loop finished.
INFO:nv_ingest.framework.orchestration.ray.primitives.ray_stat_collector:Stats collector thread stopped.
INFO:nv_ingest.framework.orchestration.ray.primitives.ray_pipeline:Scaling loop finished.
ERROR:nv_ingest.framework.orchestration.ray.primitives.ray_pipeline:An unexpected error occurred during actor shutdown: wait() expected a list of ray.ObjectRef or ray.ObjectRefGenerator, got list containing <class 'NoneType'>
Traceback (most recent call last):
  File "/opt/miniconda3/envs/rag_env/lib/python3.12/site-packages/nv_ingest/framework/orchestration/ray/util/pipeline/pipeline_runners.py", line 270, in _launch_pipeline
    time.sleep(5)
KeyboardInterrupt

During handling of the above 